[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import math
from itertools import combinations, combinations_with_replacement

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print('Using gpu: %s ' % torch.cuda.is_available())

# [像 Transformer 一样思考](https://arxiv.org/abs/2106.06981)

这里我们在不做任何训练的情况下编写我们的'玩具' GPT，用来计算直方图。对于输入序列 `<BOS>,a,a,b,a,b,c`，输出应该是 `0,3,3,2,3,2,1`，因为字母 `a` 出现了 3 次，字母 `b` 出现了 2 次，字母 `c` 出现了 1 次。每个字母被替换成它的出现次数（`<BOS>` 替换成 `0`）。

## 自注意力

首先编写你的自注意力层（暂时不用担心初始化）。


In [ ]:
class SelfAttentionLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_channels = config.n_channels
        self.key_channels = config.key_channels
        self.Query = nn.Linear(self.n_channels, self.key_channels, bias=False)
        self.Key = nn.Linear(self.n_channels, self.key_channels, bias = False)
        self.Value = nn.Linear(self.n_channels, self.n_channels, bias = False)
           
    def _init_id(self):
        self.Query.weight.data = 100*torch.eye(self.key_channels, self.n_channels)
        self.Key.weight.data = 100*torch.eye(self.key_channels,self.n_channels)
        self.Value.weight.data = torch.eye(self.key_channels,self.n_channels)        
        
    def forward(self, x): # def forward(self, x): # x (bs, T, ic)
        Q = self.Query(x) # Q = self.Query(x) # (bs, T, kc)
        K = self.Key(x)/math.sqrt(self.key_channels) # K = self.Key(x)/math.sqrt(self.key_channels) # (bs, T, kc)
        V = self.Value(x) # V = self.Value(x) # (bs, T, oc)
        A = # A = # 你的代码
        y = # y = # 你的代码
        return y, A

检查你的实现。


In [ ]:
class toy_config:
    n_channels = 3
    key_channels = 3
    
sa_toy = SelfAttentionLayer(toy_config)

In [ ]:
input = torch.randn(5,10,3)
y,A = sa_toy(input)

In [ ]:
y.shape

In [ ]:
torch.sum(A, dim=-1)

## identity GPT

我们先从一个简单的例子开始：构造恒等映射。显然，这种情况下我们可以直接用真实 transformer block 里存在的跳跃连接。但我们要忽略这些跳跃连接，改用自注意力层。在这个实操里，我们忽略 layer norm。

为了让生活更简单，我们用 `0` 编码 `<BOS>`，用 `1` 编码字母 `a`，依此类推……

如果输入序列是 `0,1,1,2,3,4,2,3,1`，我们希望输出同样的序列。用 transformer block 显然可以做到，方法如下：
- 对每个 token 取 one-hot 编码
- 把 Query 和 Key 矩阵取为 `100*Id`
- 把 Value 矩阵取为 `Id`
这样，自注意力层的输出就会和输入一样。

然后取一个前馈网络，它就是下面代码里的恒等映射：


In [ ]:
class Block_id(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = SelfAttentionLayer(config)
        self.fake_mlp = (lambda x : x)
        self.attn._init_id()

    def forward(self, x):
        x, A = self.attn(x)
        x = self.fake_mlp(x)
        return x, A

In [ ]:
nb_digits = 4
class config:
    n_channels=nb_digits+1
    key_channels=nb_digits+1

In [ ]:
bid = Block_id(config)
one_sample = torch.tensor([[0.,0.,1.,0.,0.],[0.,1.,0.,0.,0.]]).unsqueeze(0)
bid(one_sample)

现在要真正得到恒等映射，我们需要把 one-hot 编码投影回去，这可以用一个线性层（配合好的权重初始化）完成。


In [ ]:
class GPT_id(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_channels = config.n_channels
        self.tok_emb = nn.Embedding(self.n_channels,self.n_channels)
        self.block = Block_id(config)
        self.head = nn.Linear(self.n_channels, 1, bias = False)
        self._init_weights()
        
    def _init_weights(self):
        #
        # 你的代码
        #
        
    def forward(self, idx):
        x = self.tok_emb(idx)
        x, A = self.block(x)
        return self.head(x), A

In [ ]:
gid = GPT_id(config)

In [ ]:
one_sample = torch.tensor([0,1,1,2,3,4,2,3,1]).unsqueeze(0)
y, A = gid(one_sample)

In [ ]:
y == one_sample

In [ ]:
plt.imshow(A[0,:,:].cpu().data, cmap='hot', interpolation='nearest')
plt.colorbar()
plt.show()

## histogram GPT

现在我们需要改编前面的例子，编写我们的'玩具' transformer block 和'玩具' GPT 来计算直方图：
- 你需要为 Query、Key 和 Value 矩阵找到好的初始化
- 对于前馈网络，你可以用任意你喜欢的函数来假装这个 mlp。


In [ ]:
class Block_hist(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = SelfAttentionLayer(config)
        self.fake_mlp = # self.fake_mlp = # 你的代码
        self.attn._init_hist() # self.attn._init_hist() # 这需要在你的自注意力层里实现

    def forward(self, x):
        x, A = self.attn(x)
        x = self.fake_mlp(x)
        return x, A

In [ ]:
class GPT_hist(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_channels = config.n_channels
        self.tok_emb = nn.Embedding(self.n_channels,self.n_channels)
        self.block = Block_hist(config)
        self._init_weights()
        
    def _init_weights(self):
        #
        # 你的代码
        #
        
        
    def forward(self, idx):
        x = self.tok_emb(idx)
        x, A = self.block(x)
        return x, A

先恰当地选择你的配置，然后检查你的实现：


In [ ]:
gh = GPT_hist(config)

In [ ]:
one_sample = torch.tensor([0,1,1,2,3,4,2,3,1]).unsqueeze(0)
y, A = gh(one_sample)
y

In [ ]:
y.shape

In [ ]:
plt.imshow(A[0,:,:].cpu().data, cmap='hot', interpolation='nearest')
plt.colorbar()
plt.show()

# 生成你的数据集

现在，我们将用一个'微' GPT 来学习直方图任务。在此之前，我们先用'玩具' GPT 生成数据集。由于 GPT 是等变的（对输入做置换，输出也会相应地置换），我们总是可以把输入序列排好序。事实上，我们可以算出所有不同的输入，这个数量并不算太高。对于长度为 `seq_train=s`、最多 `nb_digits=n` 个数字的序列，一共有 ${s+n-1 \choose n-1}$ 种可能。现在对每种这样的序列，我们让它通过玩具 GPT 得到标签。


In [ ]:
seq_train = 30
nb_digits = 4
comb = combinations_with_replacement(range(0,seq_train+1), nb_digits-1)

def make_seq(c, seq_train):
    c_l = [0] + list(c) + [seq_train]
    len_seq = len(c_l)-1
    return [c_l[i+1]-c_l[i] for i in range(len_seq)]

l_comb =  [make_seq(c,seq_train) for c in comb]

len(l_comb)

In [ ]:
math.comb(seq_train+nb_digits-1, nb_digits-1)

In [ ]:
def make_inputs(l_comb, nb_digits=nb_digits):
    inputs = []
    for t in l_comb:
        curr = [0]
        for (i,j) in enumerate(t):
            curr += [i+1 for _ in range(j)]
        inputs.append(torch.tensor(np.array(curr)))
    return inputs

def make_loader(len_seq,nb_digits):
    comb = combinations_with_replacement(range(0,len_seq+1), nb_digits-1)
    l_comb =  [make_seq(c,len_seq) for c in comb]
    inputs = make_inputs(l_comb)
    labels = [(gh(d.unsqueeze(0))[0].squeeze(0).squeeze(1)).type(torch.LongTensor) for d in inputs]
    dataset = list(zip(inputs,labels))
    len_in = len(dataset)
    loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)
    return loader, len_in, inputs

In [ ]:
train_loader, size_train, inputs_train = make_loader(seq_train,nb_digits)

In [ ]:
size_train

In [ ]:
batch_in = next(iter(train_loader))

In [ ]:
batch_in[0].shape

In [ ]:
batch_in[1].shape

In [ ]:
batch_in[0][0]

In [ ]:
batch_in[1][0]

# 编写'微' GPT

现在我们需要编写用于学习的'微' GPT。这里的玩法是：不加任何修改地复用我们上面的 `SelfAttentionLayer`。唯一改动的部分是把写死的 `fake_mlp` 替换成一个真正的 MLP。


In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = SelfAttentionLayer(config)
        self.mlp = # self.mlp = # 你的代码

    def forward(self, x, verbose=False): # def forward(self, x, verbose=False): # x (bs, T,ic)
        #
        # 你的代码
        #
        if verbose:
            return x, A
        else:
            return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_channels = config.n_channels
        self.nb_digits = config.nb_digits
        self.tok_emb = # self.tok_emb = # 你的代码
        self.block = Block(config)
        self.head = # self.head = # 你的代码
        
    def forward(self, idx, targets=None, verbose=False):
        # idx 的形状：(bs, len)  0=bos，1...nb_digits
        # targets 的形状：(bs, len)
        #
        # 你的代码
        #
        
        loss = None
        if targets is not None:
            loss = # loss = # 你的代码
        if verbose:
            return logits, loss, A
        else:
            return logits, loss

In [ ]:
class config_gpt:
    nb_digits = nb_digits
    n_channels = 32 
    key_channels = 64 
    max_hist = seq_train+1

In [ ]:
gptmini = GPT(config_gpt)

In [ ]:
logits, _ = gptmini(batch_in[0])

In [ ]:
logits.shape

In [ ]:
_,preds = torch.max(logits,-1)

In [ ]:
preds.shape

In [ ]:
batch_in[0].shape

In [ ]:
torch.sum(preds == batch_in[1])

In [ ]:
def train_model(model, dataloader, size, epochs=1, optimizer=None):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        running_corrects = 0
        n_batch = 0
        for inputs,targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits, loss = model(inputs,targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            _,preds = torch.max(logits,-1)
           
            running_corrects += torch.true_divide(torch.sum(preds == targets.data),targets.shape[0]*targets.shape[1])
            running_loss +=  loss.data.item()
            n_batch += 1
        epoch_loss = running_loss /n_batch
        epoch_acc = running_corrects.data.item() /n_batch
        print('Loss: {:.4f} Acc: {:.4f}'.format(
                     epoch_loss, epoch_acc))

In [ ]:
gptmini = GPT(config_gpt)
gptmini = gptmini.to(device)
lr = 0.01
optimizer = torch.optim.Adam(gptmini.parameters(),lr = lr)

In [ ]:
len_train = (seq_train+1)*size_train
train_model(gptmini,train_loader,size_train,15,optimizer)

In [ ]:
lr = 0.005
optimizer = torch.optim.Adam(gptmini.parameters(),lr = lr)
train_model(gptmini,train_loader,len_train,15,optimizer)

In [ ]:
lr = 0.001
optimizer = torch.optim.Adam(gptmini.parameters(),lr = lr)
train_model(gptmini,train_loader,len_train,15,optimizer)

In [ ]:
lr = 0.0001
optimizer = torch.optim.Adam(gptmini.parameters(),lr = lr)
train_model(gptmini,train_loader,len_train,15,optimizer)

In [ ]:
one_batch = batch_in[0].to(device)
logits, loss, A = gptmini(one_batch,verbose=True)
A.shape

In [ ]:
k = 45
plt.imshow(A[k,:,:].cpu().data, cmap='hot', interpolation='nearest')
plt.colorbar()
plt.show()

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)